# Stage 2 development: isolated prompt comparisons

This notebook runs only the **20 development tasks**. It does not train a model or alter the verifier.

**Phase A (default):** direct baseline b1-v1, gold-contract b2-v1, and gold-contract b2-v2, all using agent a1-v4. This isolates the generator prompt. Select the generator using development results.

**Phase B (optional, after A):** enable `RUN_REPAIR_COMPARISON` and set the selected generator. Add agent a1-v5 while retaining the matching a1-v4 arm. Do not change generator and repair prompts simultaneously and attribute the result to either one.

The actual subprocess sandbox must load pandas 2.3.3 and numpy 2.1.3. Installing them only on PYTHONPATH is insufficient because the sandbox uses Python's isolated mode. The new runner supplies the package root explicitly and probes versions inside the actual sandbox before model loading.

Upload/push the new files first, use a GPU Kaggle session with Internet enabled, then Run All. Download `stage2_prompt_dev_bundle.zip` at completion. The old notebook, prompts and results remain unchanged. `pipeline_check` is for CPU wiring checks and is never a model-quality result.

In [ ]:
# Configuration: Phase A first. This notebook deliberately cannot select test tasks.
REPO_URL = "https://github.com/srinivas0307/lma-group-project.git"
BRANCH = "extra"
REVISION = None  # optionally pin a specific Git commit after the first clone
RUN_REPAIR_COMPARISON = False
GENERATOR_FOR_REPAIR_COMPARISON = "b2-v2"  # choose on dev after Phase A
GENERATOR_MODEL = "Qwen/Qwen2.5-3B-Instruct"
GENERATOR_4BIT = False
AGENT_MODEL = "Qwen/Qwen2.5-7B-Instruct"
AGENT_4BIT = True
BACKEND = "hf"
SEEDS = [0]
TEMPERATURE = 0.0
MAX_ATTEMPTS = 3
MAX_TOOL_CALLS = 8
TOKEN_BUDGET = 60000
WORK = "/kaggle/working"
STAGE = "dev"
assert GENERATOR_FOR_REPAIR_COMPARISON in ("b2-v1", "b2-v2")
RUNS = [("baseline1", "b2-v1", "a1-v4"),
        ("gold_contract", "b2-v1", "a1-v4"),
        ("gold_contract", "b2-v2", "a1-v4")]
if RUN_REPAIR_COMPARISON:
    RUNS += [("gold_contract", GENERATOR_FOR_REPAIR_COMPARISON, "a1-v5")]
print("Development comparisons:", RUNS)

In [ ]:
# Clone once. Reusing this directory keeps its existing commit unless REVISION is set.
import os, sys, json, subprocess, hashlib, time
from pathlib import Path
REPO = Path(WORK) / "repo_stage2_prompt_dev"
if not REPO.exists():
    clone_url = REPO_URL
    try:
        from kaggle_secrets import UserSecretsClient
        clone_url = REPO_URL.replace("https://", "https://" + UserSecretsClient().get_secret("GITHUB_TOKEN") + "@")
    except Exception:
        pass
    p = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, clone_url, str(REPO)],
                       capture_output=True, text=True)
    if p.returncode:
        raise RuntimeError(p.stderr.replace(clone_url, REPO_URL))
if REVISION:
    subprocess.run(["git", "fetch", "origin", REVISION], cwd=REPO, check=True)
    subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO, check=True)
os.chdir(REPO)
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
NEEDED = ["eval/runner_stage2_prompts.py", "src/igt/agents/code_generator_v2.py",
          "src/igt/agents/repair_agent_v5.py", "src/igt/sandbox/environment.py"]
missing = [p for p in NEEDED if not Path(p).is_file()]
assert not missing, f"Push the new files first, then use a fresh clone directory: {missing}"
assert not subprocess.check_output(["git", "status", "--porcelain"], text=True).strip(), "Use a clean committed checkout"
print("Pinned source commit:", COMMIT)

In [ ]:
# Pin data libraries separately; do not upgrade Kaggle's torch/transformers stack.
PACKAGES = Path(WORK) / "stage2_prompt_evalpkgs"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "--target", str(PACKAGES),
                "-r", "requirements.txt"], check=True)
if BACKEND == "hf":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes"], check=True)
ENV = dict(os.environ, PYTHONPATH=str(PACKAGES), TOKENIZERS_PARALLELISM="false")
probe = """import json, importlib.metadata, pandas, numpy
v = {"pandas": pandas.__version__, "numpy": numpy.__version__}
for name in ["torch", "transformers", "bitsandbytes", "peft"]:
    try: v[name] = importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError: v[name] = None
try:
    import torch
    v["cuda_available"] = torch.cuda.is_available()
    v["gpu_count"] = torch.cuda.device_count()
    v["gpus"] = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
except ImportError:
    v["cuda_available"] = False
print(json.dumps(v))
"""
p = subprocess.run([sys.executable, "-c", probe], env=ENV, capture_output=True, text=True)
assert p.returncode == 0, p.stderr
VERSIONS = json.loads(p.stdout.strip().splitlines()[-1])
assert VERSIONS["pandas"] == "2.3.3" and VERSIONS["numpy"] == "2.1.3"
assert BACKEND != "hf" or VERSIONS["cuda_available"], "Enable a Kaggle GPU accelerator"
ENV_TAG = hashlib.sha256(json.dumps(VERSIONS, sort_keys=True).encode()).hexdigest()[:12]
OUT = Path(WORK) / "results" / "stage2_prompt_dev"
CACHE = Path(WORK) / "stage2_prompt_cache" / ENV_TAG
LOGS = OUT / "logs"
LOGS.mkdir(parents=True, exist_ok=True)
print("Parent environment:", json.dumps(VERSIONS, indent=2))
print("Child sandbox versions will be verified independently by each run.")

In [ ]:
# Run all comparison arms using exactly the same models, environment and budgets.
# Existing matching runs resume; a changed environment, prompt or budget is rejected.
assert STAGE == "dev"
GEN_TAG = GENERATOR_MODEL.split("/")[-1].lower().replace(".", "_")
for preset, generator_prompt, agent_prompt in RUNS:
    label = f"{preset}__gen_{GEN_TAG}"
    log_path = LOGS / f"{label}__{generator_prompt}__{agent_prompt}.log"
    cmd = [sys.executable, "eval/runner_stage2_prompts.py", "--preset", preset, "--system", label,
           "--splits", "dev", "--backend", BACKEND,
           "--generator-prompt", generator_prompt, "--agent-prompt", agent_prompt,
           "--generator-model", GENERATOR_MODEL, "--agent-model", AGENT_MODEL,
           "--seeds", *map(str, SEEDS), "--temperature", str(TEMPERATURE),
           "--max-attempts", str(MAX_ATTEMPTS), "--max-tool-calls", str(MAX_TOOL_CALLS),
           "--token-budget", str(TOKEN_BUDGET), "--sandbox-packages", str(PACKAGES),
           "--cache-dir", str(CACHE), "--out", str(OUT)]
    if GENERATOR_4BIT: cmd.append("--generator-4bit")
    if AGENT_4BIT: cmd.append("--agent-4bit")
    start = time.time()
    print("Running", preset, generator_prompt, agent_prompt, flush=True)
    with log_path.open("a") as log:
        log.write("Command: " + json.dumps(cmd) + "\n")
        proc = subprocess.Popen(cmd, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                text=True, bufsize=1)
        for line in proc.stdout:
            log.write(line)
            log.flush()
            if line.startswith(("[seed", "Verified sandbox:", "runs in")):
                print(line, end="", flush=True)
        code = proc.wait()
        log.write(f"\nExit: {code}; minutes: {(time.time()-start)/60:.2f}\n")
    if code:
        print(log_path.read_text()[-7000:])
        raise RuntimeError(f"Comparison failed; inspect {log_path}")
    print("Completed in", round((time.time()-start)/60, 1), "minutes")

In [ ]:
# Score, record provenance and export one complete bundle. No reference enters model prompts.
p = subprocess.run([sys.executable, "eval/stage2_metrics.py", str(OUT), "dev"],
                   env=ENV, capture_output=True, text=True)
(OUT / "metrics_dev.txt").write_text(p.stdout + "\n" + p.stderr)
assert p.returncode == 0, p.stderr
print(p.stdout)
metrics = json.loads((OUT / "stage2_metrics_dev.json").read_text())
for name, scores in metrics["systems"].items():
    print(name, "initial:", scores["initial_code_accuracy"], "final:", scores["final_accuracy"],
          "silent:", scores["silent_error_rate"], "mean tokens:", scores["mean_tokens"])
if BACKEND != "hf":
    print("SCRIPTED PIPELINE CHECK ONLY: these are not model-quality results.")
source_files = set()
for folder, pattern in [("src/igt", "*.py"), ("eval", "*.py"), ("benchmark/tasks", "*.yaml"),
                        ("benchmark/prompts", "*.json")]:
    source_files.update(Path(folder).rglob(pattern))
source_files.update(Path("data") / name for name in ["adult_dev.csv", "airbnb_dev.csv",
                    "Titanic-Dataset.csv", "yellow_tripdata_2026-01.csv", "checksums.json"])
source_files.add(Path("requirements.txt"))
manifest = {str(p): hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted(source_files)}
(OUT / "source_manifest.json").write_text(json.dumps(manifest, indent=2))
for info in OUT.glob("*/run_info.json"):
    (info.parent / "git_commit.txt").write_text(COMMIT)
metadata = dict(stage="dev", commit=COMMIT, backend=BACKEND, valid_model_result=BACKEND=="hf",
                runs=RUNS, versions=VERSIONS, environment_tag=ENV_TAG,
                generator_model=GENERATOR_MODEL, generator_4bit=GENERATOR_4BIT,
                agent_model=AGENT_MODEL, agent_4bit=AGENT_4BIT,
                seeds=SEEDS, temperature=TEMPERATURE, max_attempts=MAX_ATTEMPTS,
                max_tool_calls=MAX_TOOL_CALLS, token_budget=TOKEN_BUDGET,
                sandbox_packages=str(PACKAGES), repair_comparison_enabled=RUN_REPAIR_COMPARISON)
(OUT / "bundle_info.json").write_text(json.dumps(metadata, indent=2))
import zipfile
BUNDLE = Path(WORK) / "stage2_prompt_dev_bundle.zip"
with zipfile.ZipFile(BUNDLE, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUT.rglob("*")):
        if path.is_file():
            archive.write(path, str(Path("results/stage2_prompt_dev") / path.relative_to(OUT)))
print("Download:", BUNDLE, "bytes:", BUNDLE.stat().st_size)
from IPython.display import display, FileLink
os.chdir(WORK)
display(FileLink(BUNDLE.name))

## Read the comparison before enabling Phase B

For the generator study, compare **b2-v1 / a1-v4** against **b2-v2 / a1-v4**, especially initial code accuracy. All arms use the same verified sandbox; do not use the earlier uploaded run as the only control.

After selecting the generator on dev, enable the optional agent study and compare **that generator / a1-v4** against **that generator / a1-v5**. Repeated identical configurations resume.

Keep the original development bundle and this new bundle. Do not claim that any prompt improved accuracy until the real HF results support it. Known schema gaps such as unwanted extra columns remain unchanged and must still be reported. Freeze the selected versions before test evaluation.